In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv('../../../data/processed/land_dataset_final_v3.csv')

In [3]:
X = df.drop(columns=["price_per_m2", "address_locality", "address_subdivision", "price", "longitude", "latitude", "geometry"])
# X = X.drop(columns=["mean_price_per_m2", "max_price_per_m2", "median_price_per_m2", "min_price_per_m2"])
y = df["price_per_m2"]

In [4]:
label_encoders = {}
for column in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])
    label_encoders[column] = le

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:
# Method 1: SelectKBest using f_regression (for linear relationships)
print("\nSelectKBest with f_regression:")
selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X_train, y_train)

# Get scores and p-values
scores = selector.scores_
p_values = selector.pvalues_

# Create a DataFrame to display results
feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': scores,
    'P-value': p_values
}).sort_values(by='Score', ascending=False)

print(feature_scores)



SelectKBest with f_regression:
                  Feature         Score  P-value
201             n_atm_5km  30126.319093      0.0
120  n_cafe_in_1km_to_2km  29852.874528      0.0
171    n_seven_eleven_5km  29799.556747      0.0
141            n_mart_5km  29363.855245      0.0
117            n_cafe_5km  28951.241298      0.0
..                    ...           ...      ...
210             f_disused      0.000000      1.0
217                f_road      0.000000      1.0
208            f_corridor      0.000000      1.0
207           f_bridleway      0.000000      1.0
226              f_unused      0.000000      1.0

[227 rows x 3 columns]


In [7]:

# Method 2: SelectKBest using mutual_info_regression (for non-linear relationships)
print("\nSelectKBest with mutual_info_regression:")
mi_selector = SelectKBest(score_func=mutual_info_regression, k='all')
mi_selector.fit(X_train, y_train)

mi_scores = mi_selector.scores_

mi_feature_scores = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False)

print(mi_feature_scores)


SelectKBest with mutual_info_regression:
                     Feature  MI_Score
0             address_line_2  2.273914
1                       h_id  2.060723
87   near_Royal_Palace_in_km  1.205829
123        n_gas_station_5km  1.201767
57     near_Phsar_Tmey_in_km  1.199859
..                       ...       ...
208               f_corridor  0.000000
217                   f_road  0.000000
207              f_bridleway  0.000000
210                f_disused  0.000000
226                 f_unused  0.000000

[227 rows x 2 columns]


In [8]:

# Method 3: Feature importance from Random Forest
print("\nRandom Forest Feature Importance:")
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(rf_feature_importance)

# Select top features based on the methods above
top_features = 3  # Choose how many top features you want
selected_features = feature_scores.head(top_features)['Feature'].tolist()

print(f"\nSelected top {top_features} features:", selected_features)

# Create new datasets with selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Train and evaluate model with selected features
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_selected, y_train)
y_pred = model.predict(X_test_selected)

mse = mean_squared_error(y_test, y_pred)
print(f"\nModel MSE with selected features: {mse:.4f}")

# For comparison, train model with all features
model_all = RandomForestRegressor(n_estimators=100, random_state=42)
model_all.fit(X_train, y_train)
y_pred_all = model_all.predict(X_test)

mse_all = mean_squared_error(y_test, y_pred_all)
print(f"Model MSE with all features: {mse_all:.4f}")


Random Forest Feature Importance:
                     Feature  Importance
195               n_bank_5km    0.611832
201                n_atm_5km    0.126998
0             address_line_2    0.051671
179       n_resturant_in_1km    0.032568
87   near_Royal_Palace_in_km    0.028976
..                       ...         ...
217                   f_road    0.000000
210                f_disused    0.000000
208               f_corridor    0.000000
207              f_bridleway    0.000000
226                 f_unused    0.000000

[227 rows x 2 columns]

Selected top 3 features: ['n_atm_5km', 'n_cafe_in_1km_to_2km', 'n_seven_eleven_5km']

Model MSE with selected features: 558715.6583
Model MSE with all features: 70265.6334


In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import RFECV, SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [10]:
# Method 1: Recursive Feature Elimination with Cross-Validation (RFECV)
print("\nMethod 1: RFECV with Random Forest")
estimator = RandomForestRegressor(n_estimators=100, random_state=42)
selector_rfecv = RFECV(estimator, step=1, cv=5, scoring='neg_mean_squared_error')
selector_rfecv.fit(X_train, y_train)

print("Optimal number of features:", selector_rfecv.n_features_)
print("Selected features:", X.columns[selector_rfecv.support_])

# Plot number of features vs. cross-validation scores
plt.figure()
plt.xlabel("Number of features selected")
plt.ylabel("Cross validation score (negative MSE)")
plt.plot(range(1, len(selector_rfecv.cv_results_['mean_test_score']) + 1), 
         selector_rfecv.cv_results_['mean_test_score'])
plt.show()

# Method 2: LassoCV for feature selection
print("\nMethod 2: LassoCV Feature Selection")
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)

print("Lasso alpha:", lasso.alpha_)
importance = np.abs(lasso.coef_)
selected_features_lasso = X.columns[importance > 0]
print("Selected features by Lasso:", selected_features_lasso.tolist())

# Method 3: SelectFromModel with RandomForest
print("\nMethod 3: SelectFromModel with RandomForest")
selector_sfm = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42), 
                             threshold='median')
selector_sfm.fit(X_train, y_train)
selected_features_sfm = X.columns[selector_sfm.get_support()]
print("Selected features by SelectFromModel:", selected_features_sfm.tolist())

# Compare performance
methods = {
    'All Features': X.columns,
    'RFECV': X.columns[selector_rfecv.support_],
    'LassoCV': selected_features_lasso,
    'SelectFromModel': selected_features_sfm
}

for name, features in methods.items():
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    scores = cross_val_score(model, X_train[features], y_train, 
                           cv=5, scoring='neg_mean_squared_error')
    avg_mse = -scores.mean()
    print(f"\n{name} - Cross-validated MSE: {avg_mse:.4f}")
    
    # Fit on full training set and evaluate on test set
    model.fit(X_train[features], y_train)
    test_mse = mean_squared_error(y_test, model.predict(X_test[features]))
    print(f"{name} - Test MSE: {test_mse:.4f}")

# Select the best performing method
best_method = min(methods.keys(), 
                 key=lambda x: mean_squared_error(y_test, 
                 RandomForestRegressor(n_estimators=100, random_state=42)
                 .fit(X_train[methods[x]], y_train)
                 .predict(X_test[methods[x]])))

print(f"\nBest performing method: {best_method}")
print("Selected features:", methods[best_method].tolist())


Method 1: RFECV with Random Forest


KeyboardInterrupt: 